In [36]:
import duckdb
import polars as pl
pl.Config.set_tbl_rows(100)

polars.config.Config

In [13]:
conn = duckdb.connect(database='../dados/futebol_brasileiro.duckdb')

In [39]:
conn.sql("""
WITH
jogo_a_jogo AS (
    SELECT
        A.ID_TIME,
        A.ID_TEMPORADA,
        A.NOME_CAMPEONATO,
        A.ID_RODADA,
        A.DIA,
        A.DATA_HORA_JOGO,
        A.NOME_TIME,
        A.NOME_ADVERSARIO,
        A.FLG_JOGO_EM_CASA,
        A.TREINADOR,
        A.GOLS_PRO,
        A.GOLS_PRO - A.GOLS_CONTRA AS SALDO_RESULTADO,
        A.PONTOS,
        -- saldo de gols acumulados antes da rodada
        (((A.GOLS_PRO_ACUMULADOS - A.GOLS_PRO) - (A.GOLS_CONTRA_ACUMULADOS - A.GOLS_CONTRA)) - ((B.GOLS_PRO_ACUMULADOS - B.GOLS_PRO) - (B.GOLS_CONTRA_ACUMULADOS - B.GOLS_CONTRA))) AS DIFERENCA_SALDO_GOLS_ACUMULADOS_ANTES_DA_RODADA,
        -- pontos acumulados antes da rodada
        ((A.PONTOS_ACUMULADOS - A.PONTOS) - (B.PONTOS_ACUMULADOS - B.PONTOS)) AS DIFERENCA_PONTOS_ACUMULADOS_ANTES_DA_RODADA
    FROM 
        futbr_gold.tb_fato_jogo_a_jogo AS A
    LEFT JOIN
        futbr_gold.tb_fato_jogo_a_jogo AS B
        ON A.ID_TIME_ADVERSARIO = B.ID_TIME
        AND A.ID_JOGO = B.ID_JOGO
    WHERE
        A.PONTOS IS NOT NULL
        AND A.TREINADOR IS NOT NULL
),

jogo_a_jogo_com_ajuste AS (
    SELECT
        A.*,
        ROW_NUMBER() OVER (
            PARTITION BY 
                A.ID_TIME, 
                A.ID_TEMPORADA, 
                A.NOME_CAMPEONATO,
                A.TREINADOR
            ORDER BY
                A.DATA_HORA_JOGO ASC
        ) - 1 AS ORDEM_JOGOS_TREINADOR,
        LAG(A.TREINADOR, 1) OVER (
            PARTITION BY 
                A.ID_TIME, 
                A.ID_TEMPORADA, 
                A.NOME_CAMPEONATO
            ORDER BY
                A.DATA_HORA_JOGO ASC
        ) AS TREINADOR_DATA_ANTERIOR,
        CASE
            WHEN A.TREINADOR <> TREINADOR_DATA_ANTERIOR THEN 1
            ELSE 0
        END AS FLG_MUDANCA_TREINADOR
    FROM
        jogo_a_jogo AS A
)

SELECT * FROM jogo_a_jogo_com_ajuste WHERE NOME_TIME = 'VASCO DA GAMA' AND ID_TEMPORADA = 2023
""").pl()

ID_TIME,ID_TEMPORADA,NOME_CAMPEONATO,ID_RODADA,DIA,DATA_HORA_JOGO,NOME_TIME,NOME_ADVERSARIO,FLG_JOGO_EM_CASA,TREINADOR,GOLS_PRO,SALDO_RESULTADO,PONTOS,DIFERENCA_SALDO_GOLS_ACUMULADOS_ANTES_DA_RODADA,DIFERENCA_PONTOS_ACUMULADOS_ANTES_DA_RODADA,ORDEM_JOGOS_TREINADOR,TREINADOR_DATA_ANTERIOR,FLG_MUDANCA_TREINADOR
i64,i32,str,i64,str,datetime[μs],str,str,i32,str,i32,i32,i32,"decimal[38,0]","decimal[38,0]",i64,str,i32
978,2023,"""BRASILEIRÃO SERIE A""",1,"""SÁB""",2023-04-15 21:00:00,"""VASCO DA GAMA""","""ATLÉTICO-MG""",0,"""MAURÍCIO BARBIERI""",2,1,3,0,0,0,null,0
978,2023,"""BRASILEIRÃO SERIE A""",2,"""DOM""",2023-04-23 16:00:00,"""VASCO DA GAMA""","""PALMEIRAS""",1,"""CLAUDIO MALDONADO""",2,0,1,0,0,0,"""MAURÍCIO BARBIERI""",1
978,2023,"""BRASILEIRÃO SERIE A""",3,"""SEG""",2023-05-01 20:00:00,"""VASCO DA GAMA""","""BAHIA""",1,"""MAURÍCIO BARBIERI""",0,-1,0,3,4,1,"""CLAUDIO MALDONADO""",1
978,2023,"""BRASILEIRÃO SERIE A""",4,"""SÁB""",2023-05-06 21:00:00,"""VASCO DA GAMA""","""FLUMINENSE""",0,"""MAURÍCIO BARBIERI""",1,0,1,-3,-2,2,"""MAURÍCIO BARBIERI""",0
978,2023,"""BRASILEIRÃO SERIE A""",5,"""QUI""",2023-05-11 19:00:00,"""VASCO DA GAMA""","""CORITIBA""",0,"""MAURÍCIO BARBIERI""",1,0,1,8,4,3,"""MAURÍCIO BARBIERI""",0
978,2023,"""BRASILEIRÃO SERIE A""",6,"""DOM""",2023-05-14 16:00:00,"""VASCO DA GAMA""","""SANTOS""",1,"""MAURÍCIO BARBIERI""",0,-1,0,-2,-1,4,"""MAURÍCIO BARBIERI""",0
978,2023,"""BRASILEIRÃO SERIE A""",7,"""SÁB""",2023-05-20 18:30:00,"""VASCO DA GAMA""","""SÃO PAULO""",0,"""MAURÍCIO BARBIERI""",2,-2,0,-5,-3,5,"""MAURÍCIO BARBIERI""",0
978,2023,"""BRASILEIRÃO SERIE A""",8,"""SÁB""",2023-05-27 16:00:00,"""VASCO DA GAMA""","""FORTALEZA""",0,"""MAURÍCIO BARBIERI""",0,-2,0,-7,-4,6,"""MAURÍCIO BARBIERI""",0
978,2023,"""BRASILEIRÃO SERIE A""",9,"""SEG""",2023-06-05 20:00:00,"""VASCO DA GAMA""","""FLAMENGO""",1,"""MAURÍCIO BARBIERI""",1,-3,0,-9,-7,7,"""MAURÍCIO BARBIERI""",0


In [25]:
-3 - (-12)

9

In [5]:
conn.close()